In [1]:
#Importing all the required libraries

# Document loading & chunking
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings #Only install a class from huggingface

# Vector store
from langchain_community.vectorstores import Chroma

# LLM
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
import os

C:\Users\manog\AppData\Local\Temp\ipykernel_54316\995767986.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
#Loading the book
documents_map = {
    "DIFC_Court_Rules.pdf": "DIFC Court Rules",
    "DFSA_Mkt_Rules_24-25.pdf": "DFSA Markets Rules",
    "DFSA_Gen_module.pdf": "DFSA General Module",
    "DFSA_Gen_Appendix_1.pdf": "DFSA GEN Appendix 1",
    "DFSA_Gen_Appendix_2.pdf": "DFSA GEN Appendix 2",
    "DFSA_Gen_Appendix_3.pdf": "DFSA GEN Appendix 3",
    "DFSA_Gen_Appendix_4.pdf": "DFSA GEN Appendix 4",
    "UAE_Federal_Aml LAW.pdf": "UAE Federal AML Law",
    "DIFC_Data_Protection_Law.pdf": "DIFC Data Protection Law",
}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\nArticle", "\nSection", "\n\n", "\n"]
)

all_chunks = []

for filename, doc_name in documents_map.items():
    print(f"Loading {doc_name}...")
    loader = PyPDFLoader(filename)
    pages = loader.load()
    chunks = splitter.split_documents(pages)
    
    # Tag every chunk with which document it came from
    for chunk in chunks:
        chunk.metadata["source_doc"] = doc_name
    
    all_chunks.extend(chunks)
    print(f"  → {len(chunks)} chunks")

print(f"\nTotal chunks across all documents: {len(all_chunks)}")

Loading DIFC Court Rules...
  → 2404 chunks
Loading DFSA Markets Rules...
  → 979 chunks
Loading DFSA General Module...
  → 1145 chunks
Loading DFSA GEN Appendix 1...
  → 73 chunks
Loading DFSA GEN Appendix 2...
  → 39 chunks
Loading DFSA GEN Appendix 3...
  → 231 chunks
Loading DFSA GEN Appendix 4...
  → 67 chunks
Loading UAE Federal AML Law...
  → 112 chunks
Loading DIFC Data Protection Law...
  → 372 chunks

Total chunks across all documents: 5422


In [3]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    persist_directory="./difc_chroma_db"
)

print("All documents embedded and stored.")

C:\Users\manog\AppData\Local\Temp\ipykernel_54316\548255281.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

All documents embedded and stored.


In [4]:
# Load existing vectorstore (no re-embedding needed)
vectorstore = Chroma(
    persist_directory="./difc_chroma_db",
    embedding_function=embeddings
)

# Set up Groq
llm = ChatGroq(
    api_key=os.environ["GROQ_API_KEY"],
    model_name="llama-3.1-8b-instant"
)

# Your question
query = "What happens if a defendant fails to respond to a claim?"

# Retrieve relevant chunks
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
relevant_chunks = retriever.invoke(query)

# Build prompt
context = "\n\n".join([
    f"[Page {doc.metadata.get('page', 'N/A')}]: {doc.page_content}"
    for doc in relevant_chunks
])

prompt = f"""You are a compliance assistant for DIFC regulations. 
Answer the question using ONLY the context below.
Cite the page number for every point you make.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {query}
"""

response = llm.invoke(prompt)
print(response.content)

C:\Users\manog\AppData\Local\Temp\ipykernel_54316\694605798.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


According to the context, I can provide the following information:

If a defendant has not satisfied the claim (page 73), they have not returned an admission to the claimant under Rule 15.14 (page 73), or they have not filed an admission with the Court under Rule 15.24 (page 73), and they were served with the claim outside the jurisdiction but have not acknowledged service, the claimant must establish that the claim is one that the Court has power to hear and decide (page 73).

However, the context does not directly address the consequences of a defendant failing to respond to a claim. But, according to [Page 304] 35.14, if the claimant does not attend the trial, the Court may strike out his claim and any defence to counterclaim.
